# Mechanical and ultrasonic data integration

This notebook combines the reduced mechanical record with the synchronized waveform-pick table produced by **Mech Wave Analyzer v3**.

The workflow is:

1. import and inspect both files;
2. verify synchronization using both exported time and row coordinates;
3. select one arrival-picking method;
4. calculate P- and S-wave travel times, velocities and \(V_P/V_S\);
5. open an interactive pop-up plot with selectable mechanical axes and a second acoustic y axis.

The original files are never modified. All calculations are stored in new DataFrames.

## 0. Environment

Run the notebook in the same Conda environment used for the waveform analyzer. If necessary, install the packages below from Anaconda Prompt or Terminal:

```bash
conda activate mechwave
python -m pip install numpy pandas matplotlib PyQt6 jupyter
```

The final plotting cell uses the Qt backend so the selector opens in a separate interactive window.

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

COLORS = {
    'ink': '#263238',
    'teal': '#167C80',
    'orange': '#D9792B',
    'blue': '#2B5C8F',
    'red': '#C73E3A',
    'green': '#2E7D32',
    'purple': '#7A5195',
    'gray': '#6B7280',
}

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 140)

## 1. Analysis settings

Edit this cell before running the notebook on a new experiment.

- `ARRIVAL_METHOD` may be `cross_correlation`, `sta_lta`, `manual`, or `best_available`.
- `best_available` uses manual first, then cross-correlation, then STA/LTA **for each phase independently**. Mixing methods is convenient for inspection but should be avoided in a final interpretation unless it is explicitly documented.
- Travel time is the picked arrival minus the independently measured system delay.
- The initial path length must be the physical transmitter-to-receiver distance through the specimen—not simply the nominal specimen dimension unless they are the same.
- If propagation is axial, the current path length can be corrected using measured specimen shortening.

In [2]:
# Input filenames. The loader searches the notebook folder and common nearby folders.
MECHANICAL_FILENAME = 'triax_try_sandstone_acousto_mech.txt'
ACOUSTIC_FILENAME = 'synched_waveform_picks.csv'

# Pick source: 'cross_correlation', 'sta_lta', 'manual', or 'best_available'.
ARRIVAL_METHOD = 'cross_correlation'
MIN_CC_COEFFICIENT = 0.50

# Geometry and timing calibration.
P_INITIAL_PATH_LENGTH_MM = 50.0
S_INITIAL_PATH_LENGTH_MM = 50.0
P_SYSTEM_DELAY_US = 0.0
S_SYSTEM_DELAY_US = 0.0

# Set True only when propagation is along the shortening direction.
APPLY_AXIAL_SHORTENING_TO_PATH = True
SHORTENING_COLUMN = 'axial_shortening_corrected_mm'

# Set False for automated/headless testing. Keep True during normal notebook use.
OPEN_POPUP = os.environ.get('MECH_ACOUSTIC_HEADLESS', '0') != '1'

## 2. Import both data files

The mechanical output is a tab-separated table. Its first row contains channel names and its second row contains units, so the second row is read separately and then skipped when loading the numerical data.

The acoustic output is a standard comma-separated CSV file.

In [3]:
def locate_file(filename):
    """Find an input file without hard-coding one computer's absolute path."""
    candidates = [
        Path.cwd() / filename,
        Path.cwd() / 'upload' / filename,
        Path.cwd().parent / filename,
        Path.cwd().parent / 'upload' / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    searched = '\n'.join(f'  - {path}' for path in candidates)
    raise FileNotFoundError(f'Could not find {filename}. Searched:\n{searched}')


def read_mechanical_output(path):
    """Read channel names, the units row, and numerical mechanical data."""
    with open(path, 'r', encoding='utf-8-sig') as stream:
        header = stream.readline().rstrip('\n\r').split('\t')
        unit_values = stream.readline().rstrip('\n\r').split('\t')
    unit_values += [''] * (len(header) - len(unit_values))
    units = dict(zip(header, unit_values[:len(header)]))

    frame = pd.read_csv(path, sep='\t', skiprows=[1], encoding='utf-8-sig')
    for column in frame.columns:
        frame[column] = pd.to_numeric(frame[column], errors='coerce')
    return frame, units


mechanical_path = locate_file(MECHANICAL_FILENAME)
acoustic_path = locate_file(ACOUSTIC_FILENAME)

mechanical, mechanical_units = read_mechanical_output(mechanical_path)
acoustic_raw = pd.read_csv(acoustic_path)

print(f'Mechanical file: {mechanical_path.name} — {len(mechanical):,} rows, {len(mechanical.columns)} columns')
print(f'Acoustic file:   {acoustic_path.name} — {len(acoustic_raw):,} traces, {len(acoustic_raw.columns)} columns')

display(pd.DataFrame({
    'mechanical_column': mechanical.columns,
    'unit': [mechanical_units.get(column, '') for column in mechanical.columns],
    'records': len(mechanical),
}).head(30))

display(acoustic_raw.head())

Mechanical file: triax_try_sandstone_acousto_mech.txt — 6,180 rows, 21 columns
Acoustic file:   synched_waveform_picks.csv — 2,387 traces, 13 columns


,mechanical_column,unit,records
0,time_s,s,6180
1,sync,V,6180
2,record,#,6180
3,vertical_load_raw,raw ADC value,6180
4,vertical_load_calibrated,kN,6180
5,horizontal_load_raw,raw ADC value,6180
6,horizontal_load_calibrated,kN,6180
7,vertical_displacement_raw,raw ADC value,6180
8,vertical_displacement_calibrated,µm,6180
9,horizontal_displacement_raw,raw ADC value,6180


,Interval_Num,File_Name,Trace_Index,Mech_Time_s,Mech_Row_Index,P_Manual_us,S_Manual_us,P_Auto_us,S_Auto_us,P_CC_us,S_CC_us,P_CC_Coefficient,S_CC_Coefficient
0,1,first_sand.bscan.tsv,0,42.400000,424,NaN,NaN,47.8,53.9,NaN,NaN,NaN,NaN
1,1,first_sand.bscan.tsv,1,42.501174,425,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,first_sand.bscan.tsv,2,42.602349,426,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,first_sand.bscan.tsv,3,42.703523,427,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,first_sand.bscan.tsv,4,42.804698,428,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Synchronization audit

The waveform analyzer exports two links to the mechanical record:

- `Mech_Time_s`: the interpolated experiment time assigned to each waveform;
- `Mech_Row_Index`: the corresponding zero-based mechanical row.

The notebook uses `Mech_Time_s` for interpolation because it preserves the fractional position of every trace. `Mech_Row_Index` is retained as an independent check. With regularly sampled mechanical data, the difference between the exported time and the time at the rounded row should be no larger than approximately half one mechanical sampling interval.

In [4]:
REQUIRED_MECHANICAL = {'time_s', 'record'}
REQUIRED_ACOUSTIC = {
    'Interval_Num', 'File_Name', 'Trace_Index', 'Mech_Time_s', 'Mech_Row_Index',
    'P_Manual_us', 'S_Manual_us', 'P_Auto_us', 'S_Auto_us',
    'P_CC_us', 'S_CC_us', 'P_CC_Coefficient', 'S_CC_Coefficient',
}

missing_mechanical = REQUIRED_MECHANICAL.difference(mechanical.columns)
missing_acoustic = REQUIRED_ACOUSTIC.difference(acoustic_raw.columns)
if missing_mechanical:
    raise KeyError(f'Mechanical file is missing: {sorted(missing_mechanical)}')
if missing_acoustic:
    raise KeyError(f'Acoustic file is missing: {sorted(missing_acoustic)}')

if not mechanical['time_s'].is_monotonic_increasing:
    raise ValueError('Mechanical time_s must increase monotonically for synchronization.')

acoustic = acoustic_raw.copy()
numeric_acoustic = [
    column for column in acoustic.columns
    if column not in {'File_Name'}
]
for column in numeric_acoustic:
    acoustic[column] = pd.to_numeric(acoustic[column], errors='coerce')

row_indices = acoustic['Mech_Row_Index'].round().astype('Int64')
rows_valid = row_indices.between(0, len(mechanical) - 1).fillna(False)
if not rows_valid.all():
    bad = acoustic.loc[~rows_valid, ['File_Name', 'Trace_Index', 'Mech_Row_Index']]
    raise IndexError(f'Acoustic rows outside mechanical table bounds:\n{bad.head()}')

row_times = mechanical.iloc[row_indices.astype(int).to_numpy()]['time_s'].to_numpy(float)
time_mismatch_s = np.abs(row_times - acoustic['Mech_Time_s'].to_numpy(float))
mechanical_dt_s = float(np.nanmedian(np.diff(mechanical['time_s'].to_numpy(float))))

time_inside = acoustic['Mech_Time_s'].between(
    mechanical['time_s'].min(), mechanical['time_s'].max()
)
if not time_inside.all():
    raise ValueError('Some acoustic times fall outside the mechanical time range.')

sync_report = pd.DataFrame({
    'check': [
        'Mechanical time range', 'Acoustic time range', 'Mechanical median sampling interval',
        'Maximum time versus rounded-row mismatch', 'Expected half-sample tolerance',
        'Acoustic rows within mechanical bounds',
    ],
    'value': [
        f"{mechanical['time_s'].min():.3f} to {mechanical['time_s'].max():.3f} s",
        f"{acoustic['Mech_Time_s'].min():.3f} to {acoustic['Mech_Time_s'].max():.3f} s",
        f'{mechanical_dt_s:.6g} s',
        f'{np.nanmax(time_mismatch_s):.6g} s',
        f'{0.5 * mechanical_dt_s:.6g} s',
        bool(rows_valid.all()),
    ]
})
display(sync_report)

if np.nanmax(time_mismatch_s) > 0.5 * mechanical_dt_s + 1e-9:
    print('WARNING: exported time and row coordinates differ by more than half a mechanical sample.')
else:
    print('Synchronization check passed: time and row coordinates are mutually consistent.')

,check,value
0,Mechanical time range,0.000 to 617.900 s
1,Acoustic time range,42.400 to 584.000 s
2,Mechanical median sampling interval,0.1 s
3,Maximum time versus rounded-row mismatch,0.05 s
4,Expected half-sample tolerance,0.05 s
5,Acoustic rows within mechanical bounds,True


Synchronization check passed: time and row coordinates are mutually consistent.


## 4. Flight time and velocity calculation

For each phase, the corrected specimen travel time is

\[
t_{P}=t_{P,\,pick}-t_{P,\,system}
\qquad
t_{S}=t_{S,\,pick}-t_{S,\,system}
\]

where the system delay is measured independently using the appropriate calibration procedure. It is not estimated by this notebook.

For a path length \(L\), velocity is

\[
V=\frac{L}{t}
\]

With \(L\) in millimetres and \(t\) in microseconds:

\[
V\,[\mathrm{m\,s^{-1}}]
=1000\,\frac{L\,[\mathrm{mm}]}{t\,[\mu\mathrm{s}]}
\]

If the wave path is axial and shortening is positive in compression, the instantaneous path length is

\[
L_i=L_0-\Delta L_i
\]

Finally,

\[
\frac{V_P}{V_S}=\frac{L_P/t_P}{L_S/t_S}
\]

When the P and S path lengths are identical, \(V_P/V_S=t_S/t_P\).

In [5]:
PICK_COLUMNS = {
    'manual': ('P_Manual_us', 'S_Manual_us'),
    'sta_lta': ('P_Auto_us', 'S_Auto_us'),
    'cross_correlation': ('P_CC_us', 'S_CC_us'),
}


def first_available_with_method(frame, phase):
    """Choose manual, then CC, then STA/LTA and record which method supplied each value."""
    candidates = [
        ('manual', f'{phase}_Manual_us'),
        ('cross_correlation', f'{phase}_CC_us'),
        ('sta_lta', f'{phase}_Auto_us'),
    ]
    values = pd.Series(np.nan, index=frame.index, dtype=float)
    methods = pd.Series(pd.NA, index=frame.index, dtype='object')
    for method, column in candidates:
        available = values.isna() & frame[column].notna()
        values.loc[available] = frame.loc[available, column]
        methods.loc[available] = method
    return values, methods


def interpolate_mechanical(column, times_s, mechanical_frame=mechanical):
    """Interpolate one numerical mechanical column at waveform times."""
    if column not in mechanical_frame.columns:
        raise KeyError(f'Mechanical column not found: {column}')
    values = pd.to_numeric(mechanical_frame[column], errors='coerce').to_numpy(float)
    times = mechanical_frame['time_s'].to_numpy(float)
    valid = np.isfinite(times) & np.isfinite(values)
    if valid.sum() < 2:
        return np.full(len(times_s), np.nan)
    return np.interp(np.asarray(times_s, dtype=float), times[valid], values[valid])


def calculate_acoustic_quantities(frame):
    result = frame.copy()

    if ARRIVAL_METHOD == 'best_available':
        result['P_pick_us'], result['P_pick_method'] = first_available_with_method(result, 'P')
        result['S_pick_us'], result['S_pick_method'] = first_available_with_method(result, 'S')
    elif ARRIVAL_METHOD in PICK_COLUMNS:
        p_column, s_column = PICK_COLUMNS[ARRIVAL_METHOD]
        result['P_pick_us'] = result[p_column]
        result['S_pick_us'] = result[s_column]
        result['P_pick_method'] = ARRIVAL_METHOD
        result['S_pick_method'] = ARRIVAL_METHOD
    else:
        raise ValueError(f'Unknown ARRIVAL_METHOD: {ARRIVAL_METHOD!r}')

    # Apply an explicit quality threshold only to CC-derived picks.
    p_is_cc = result['P_pick_method'].eq('cross_correlation')
    s_is_cc = result['S_pick_method'].eq('cross_correlation')
    result.loc[p_is_cc & (result['P_CC_Coefficient'] < MIN_CC_COEFFICIENT), 'P_pick_us'] = np.nan
    result.loc[s_is_cc & (result['S_CC_Coefficient'] < MIN_CC_COEFFICIENT), 'S_pick_us'] = np.nan

    result['P_flight_time_us'] = result['P_pick_us'] - float(P_SYSTEM_DELAY_US)
    result['S_flight_time_us'] = result['S_pick_us'] - float(S_SYSTEM_DELAY_US)
    result.loc[result['P_flight_time_us'] <= 0, 'P_flight_time_us'] = np.nan
    result.loc[result['S_flight_time_us'] <= 0, 'S_flight_time_us'] = np.nan

    if APPLY_AXIAL_SHORTENING_TO_PATH:
        shortening = interpolate_mechanical(SHORTENING_COLUMN, result['Mech_Time_s'])
    else:
        shortening = np.zeros(len(result), dtype=float)
    result['path_shortening_mm'] = shortening
    result['P_path_length_mm'] = float(P_INITIAL_PATH_LENGTH_MM) - shortening
    result['S_path_length_mm'] = float(S_INITIAL_PATH_LENGTH_MM) - shortening

    invalid_path = (result['P_path_length_mm'] <= 0) | (result['S_path_length_mm'] <= 0)
    if invalid_path.any():
        raise ValueError('Calculated path length is non-positive. Check geometry and shortening sign.')

    result['Vp_m_per_s'] = (
        1000.0 * result['P_path_length_mm'] / result['P_flight_time_us']
    )
    result['Vs_m_per_s'] = (
        1000.0 * result['S_path_length_mm'] / result['S_flight_time_us']
    )
    result['Vp_over_Vs'] = result['Vp_m_per_s'] / result['Vs_m_per_s']
    return result


acoustic_analysis = calculate_acoustic_quantities(acoustic)

availability = pd.DataFrame({
    'quantity': ['P flight time', 'S flight time', 'Vp', 'Vs', 'Vp/Vs'],
    'valid_records': [
        acoustic_analysis['P_flight_time_us'].notna().sum(),
        acoustic_analysis['S_flight_time_us'].notna().sum(),
        acoustic_analysis['Vp_m_per_s'].notna().sum(),
        acoustic_analysis['Vs_m_per_s'].notna().sum(),
        acoustic_analysis['Vp_over_Vs'].notna().sum(),
    ],
    'total_waveforms': len(acoustic_analysis),
})

print(f'Arrival method: {ARRIVAL_METHOD}')
print(f'P/S initial path lengths: {P_INITIAL_PATH_LENGTH_MM:g} / {S_INITIAL_PATH_LENGTH_MM:g} mm')
print(f'P/S system delays: {P_SYSTEM_DELAY_US:g} / {S_SYSTEM_DELAY_US:g} µs')
print(f'Apply axial shortening correction: {APPLY_AXIAL_SHORTENING_TO_PATH}')
display(availability)

summary_columns = [
    'P_flight_time_us', 'S_flight_time_us', 'Vp_m_per_s', 'Vs_m_per_s', 'Vp_over_Vs'
]
display(acoustic_analysis[summary_columns].describe())

Arrival method: cross_correlation
P/S initial path lengths: 50 / 50 mm
P/S system delays: 0 / 0 µs
Apply axial shortening correction: True


,quantity,valid_records,total_waveforms
0,P flight time,284,2387
1,S flight time,290,2387
2,Vp,284,2387
3,Vs,290,2387
4,Vp/Vs,284,2387


,P_flight_time_us,S_flight_time_us,Vp_m_per_s,Vs_m_per_s,Vp_over_Vs
count,284.000000,290.000000,284.000000,290.000000,284.000000
mean,42.545775,64.815172,1152.369305,756.457148,1.522489
std,0.188587,0.556102,4.255601,6.744309,0.014990
min,41.900000,64.500000,1141.811354,711.558258,1.504651
25%,42.400000,64.500000,1149.145401,754.866560,1.512881
50%,42.500000,64.600000,1153.395380,759.711365,1.517647
75%,42.700000,64.900000,1155.197214,760.143929,1.527059
max,43.000000,68.800000,1168.556192,760.699756,1.580189


### Interpretation check

Before plotting, inspect the counts and ranges above.

- Blank velocity values are expected wherever the selected method has no accepted arrival.
- Do not interpolate across missing arrivals automatically; first determine why they are missing.
- Large discontinuities may reflect real damage evolution, but can also indicate a cycle skip, a change in waveform polarity, or an inappropriate master waveform.
- If P and S were measured along different paths, enter their path lengths separately and do not interpret the time ratio alone as `Vp/Vs`.

## 5. Interactive mechanical and acoustic plot

Running the next cell opens a separate window.

- Use the three compact **drop-down menus** to choose the mechanical X variable, mechanical Y variable and acoustic display.
- The mechanical quantity is plotted on the primary left axis.
- The acoustic quantity is always plotted on the secondary right axis. Its scale, label and unit are recalculated whenever the acoustic selection changes. P and S may be shown together because they share units; the ratio is kept on its own dimensionless axis.
- Acoustic points are positioned on the selected x coordinate by interpolation at `Mech_Time_s`.
- Separate waveform files are drawn as separate line segments, preventing the plot from connecting across acquisition gaps.
- Hover over either the mechanical curve or acoustic points to display coordinates.

Close the window when you have finished. Re-run the cell to open it again.

In [6]:
ACOUSTIC_DISPLAYS = {
    'P flight time': (['P_flight_time_us'], 'Flight time [µs]'),
    'S flight time': (['S_flight_time_us'], 'Flight time [µs]'),
    'P and S flight times': (['P_flight_time_us', 'S_flight_time_us'], 'Flight time [µs]'),
    'Vp': (['Vp_m_per_s'], 'Velocity [m/s]'),
    'Vs': (['Vs_m_per_s'], 'Velocity [m/s]'),
    'Vp and Vs': (['Vp_m_per_s', 'Vs_m_per_s'], 'Velocity [m/s]'),
    'Vp/Vs ratio': (['Vp_over_Vs'], 'Vp/Vs [–]'),
}

ACOUSTIC_STYLE = {
    'P_flight_time_us': ('P flight time', COLORS['red'], 'o'),
    'S_flight_time_us': ('S flight time', COLORS['green'], 's'),
    'Vp_m_per_s': ('Vp', COLORS['red'], 'o'),
    'Vs_m_per_s': ('Vs', COLORS['green'], 's'),
    'Vp_over_Vs': ('Vp/Vs', COLORS['purple'], 'D'),
}


def open_mechanical_acoustic_explorer(mechanical_frame, acoustic_frame, units=None):
    """Open a compact Qt window with drop-down selectors above a large plot."""
    from PyQt6.QtWidgets import (
        QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout,
        QLabel, QComboBox, QPushButton,
    )
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg as FigureCanvas
    from matplotlib.backends.backend_qtagg import NavigationToolbar2QT as NavigationToolbar

    units = units or {}
    numeric_columns = [
        column for column in mechanical_frame.columns
        if pd.api.types.is_numeric_dtype(mechanical_frame[column])
        and not column.endswith('_raw')
    ]
    if len(numeric_columns) < 2:
        raise ValueError('At least two numerical mechanical columns are required.')

    preferred_x = 'time_s' if 'time_s' in numeric_columns else numeric_columns[0]
    preferred_y = (
        'axial_stress_MPa' if 'axial_stress_MPa' in numeric_columns
        else numeric_columns[1]
    )
    acoustic_options = list(ACOUSTIC_DISPLAYS)
    preferred_acoustic = 'P and S flight times'

    # Reuse Jupyter's Qt application, then build a dedicated analysis window.
    app = QApplication.instance()
    if app is None:
        app = QApplication([])
    window = QMainWindow()
    window.setWindowTitle('Mechanical and ultrasonic data explorer')
    window.resize(1450, 850)

    central_widget = QWidget()
    main_layout = QVBoxLayout(central_widget)
    main_layout.setContentsMargins(8, 8, 8, 8)
    control_layout = QHBoxLayout()

    def make_combo(label_text, values, selected, minimum_width):
        label = QLabel(label_text)
        combo = QComboBox()
        combo.addItems(values)
        combo.setCurrentText(selected)
        combo.setMinimumWidth(minimum_width)
        control_layout.addWidget(label)
        control_layout.addWidget(combo)
        return combo

    x_selector = make_combo('X variable:', numeric_columns, preferred_x, 230)
    y_selector = make_combo('Mechanical Y:', numeric_columns, preferred_y, 250)
    acoustic_selector = make_combo(
        'Acoustic right axis:', acoustic_options, preferred_acoustic, 210
    )
    reset_button = QPushButton('Reset view')
    reset_button.setMinimumWidth(105)
    control_layout.addSpacing(8)
    control_layout.addWidget(reset_button)
    control_layout.addStretch()

    fig = Figure(figsize=(12.5, 7.0))
    canvas = FigureCanvas(fig)
    toolbar = NavigationToolbar(canvas, window)
    plot_ax = fig.add_subplot(111)
    acoustic_ax = plot_ax.twinx()

    main_layout.addLayout(control_layout)
    main_layout.addWidget(toolbar)
    main_layout.addWidget(canvas, stretch=1)
    window.setCentralWidget(central_widget)

    state = {
        'x': preferred_x,
        'y': preferred_y,
        'acoustic': preferred_acoustic,
        'hover_artists': [],
        'annotations': [],
    }

    def axis_label(column):
        unit = units.get(column, '')
        return f'{column} [{unit}]' if unit else column

    def x_at_waveform_times(column):
        if column == 'time_s':
            return acoustic_frame['Mech_Time_s'].to_numpy(float)
        return interpolate_mechanical(column, acoustic_frame['Mech_Time_s'], mechanical_frame)

    def redraw():
        plot_ax.clear()
        acoustic_ax.clear()
        state['hover_artists'] = []
        state['annotations'] = []

        mechanical_line, = plot_ax.plot(
            mechanical_frame[state['x']], mechanical_frame[state['y']],
            color=COLORS['teal'], lw=1.45, label=state['y'], zorder=2
        )
        mechanical_line.set_picker(5)
        state['hover_artists'].append((mechanical_line, plot_ax, 'mechanical'))

        columns, right_label = ACOUSTIC_DISPLAYS[state['acoustic']]
        acoustic_x = x_at_waveform_times(state['x'])
        for column in columns:
            label, color, marker = ACOUSTIC_STYLE[column]
            first_segment = True
            for _, group in acoustic_frame.groupby(['Interval_Num', 'File_Name'], sort=False):
                indices = group.index.to_numpy()
                x_values = acoustic_x[indices]
                y_values = acoustic_frame.loc[indices, column].to_numpy(float)
                valid = np.isfinite(x_values) & np.isfinite(y_values)
                if not valid.any():
                    continue
                line, = acoustic_ax.plot(
                    x_values[valid], y_values[valid], color=color, marker=marker,
                    markersize=3.8, lw=1.0, alpha=0.90,
                    label=label if first_segment else '_nolegend_', zorder=3
                )
                line.set_picker(5)
                state['hover_artists'].append((line, acoustic_ax, label))
                first_segment = False

        plot_ax.set_xlabel(axis_label(state['x']))
        plot_ax.set_ylabel(axis_label(state['y']), color=COLORS['teal'])
        acoustic_ax.set_ylabel(right_label, color=COLORS['orange'])
        # Reassert the axis sides after clear(), which resets axis properties.
        plot_ax.yaxis.set_label_position('left')
        plot_ax.yaxis.tick_left()
        acoustic_ax.yaxis.set_label_position('right')
        acoustic_ax.yaxis.tick_right()
        plot_ax.tick_params(axis='y', colors=COLORS['teal'])
        acoustic_ax.tick_params(axis='y', colors=COLORS['orange'])
        plot_ax.spines['left'].set_color(COLORS['teal'])
        acoustic_ax.spines['right'].set_color(COLORS['orange'])
        acoustic_ax.spines['right'].set_visible(True)
        acoustic_ax.relim()
        acoustic_ax.autoscale_view()
        plot_ax.grid(alpha=0.22)
        plot_ax.set_title(
            f"{state['y']} with {state['acoustic']} — synchronized by experiment time",
            color=COLORS['ink'], pad=28
        )

        handles_1, labels_1 = plot_ax.get_legend_handles_labels()
        handles_2, labels_2 = acoustic_ax.get_legend_handles_labels()
        plot_ax.legend(
            handles_1 + handles_2, labels_1 + labels_2,
            loc='upper center', bbox_to_anchor=(0.5, 1.02), ncol=3,
            frameon=True, fontsize=9
        )

        for axis, edge_color in ((plot_ax, COLORS['teal']), (acoustic_ax, COLORS['orange'])):
            annotation = axis.annotate(
                '', xy=(0, 0), xytext=(14, 14), textcoords='offset points',
                bbox=dict(boxstyle='round,pad=0.35', facecolor='white', edgecolor=edge_color, alpha=0.96),
                arrowprops=dict(arrowstyle='->', color=edge_color), zorder=10
            )
            annotation.set_visible(False)
            state['annotations'].append(annotation)

        fig.subplots_adjust(left=0.085, right=0.90, bottom=0.11, top=0.88)
        canvas.draw_idle()

    def choose_x(value):
        state['x'] = value
        redraw()

    def choose_y(value):
        state['y'] = value
        redraw()

    def choose_acoustic(value):
        state['acoustic'] = value
        redraw()

    def reset_view(_event):
        plot_ax.relim()
        plot_ax.autoscale_view()
        acoustic_ax.relim()
        acoustic_ax.autoscale_view()
        for annotation in state['annotations']:
            annotation.set_visible(False)
        canvas.draw_idle()

    def show_coordinates(event):
        found = False
        was_visible = any(annotation.get_visible() for annotation in state['annotations'])
        for annotation in state['annotations']:
            annotation.set_visible(False)
        for artist, axis, label in reversed(state['hover_artists']):
            contains, information = artist.contains(event)
            if not contains or not len(information.get('ind', [])):
                continue
            point_index = int(information['ind'][0])
            x_value = np.asarray(artist.get_xdata())[point_index]
            y_value = np.asarray(artist.get_ydata())[point_index]
            annotation = state['annotations'][0 if axis is plot_ax else 1]
            annotation.xy = (x_value, y_value)
            annotation.set_text(f'{label}\nx: {x_value:.6g}\ny: {y_value:.6g}')
            annotation.set_visible(True)
            found = True
            break
        if found or was_visible:
            canvas.draw_idle()

    x_selector.currentTextChanged.connect(choose_x)
    y_selector.currentTextChanged.connect(choose_y)
    acoustic_selector.currentTextChanged.connect(choose_acoustic)
    reset_button.clicked.connect(reset_view)
    hover_connection = canvas.mpl_connect('motion_notify_event', show_coordinates)
    redraw()
    window.show()

    # Keep the Qt controls, canvas and callbacks alive while the window is open.
    window._combined_explorer_objects = (
        app, fig, canvas, plot_ax, acoustic_ax, x_selector, y_selector,
        acoustic_selector, reset_button, hover_connection,
    )
    return window


if OPEN_POPUP:
    get_ipython().run_line_magic('matplotlib', 'qt')
    combined_explorer = open_mechanical_acoustic_explorer(
        mechanical, acoustic_analysis, units=mechanical_units
    )
else:
    print('Headless mode: pop-up plot was not opened.')

## Resulting analysis table

`acoustic_analysis` retains every column from the waveform-pick export and adds:

- selected P and S picks and their method;
- corrected P and S flight times;
- time-dependent P and S path lengths;
- `Vp_m_per_s`, `Vs_m_per_s`, and `Vp_over_Vs`.

The synchronized plot calculates the selected mechanical x coordinate at every waveform time without changing the original mechanical table. This notebook intentionally stops here so the data and assumptions can be checked before smoothing, fitting, segmenting the experiment, or exporting a final combined file.